# Accelerating inference for GPT-Neo with DeepSpeed
This notebook is a companion of chapter 4 of the "Domain Specific LLMs in Action" book, author Guglielmo Iozzia, [Manning Publications](https://www.manning.com/), 2024.  
The code in this notebook is to introduce readers to the [DeepSpeed](https://github.com/microsoft/DeepSpeed) library to accelerate inference for the [GPT-Neo model](https://github.com/EleutherAI/gpt-neo) for text generation tasks. It can be executed in the Colab free tier with hardware acceleration (GPU).  
More details about the code can be found in the book's chapter.

Install the missing dependencies in the Colab VM (DeepSpeed and HF's Accelerate only).

In [3]:
!pip install deepspeed accelerate

Before loading the model, let's define a custom function to be used for benchmarking (latency measurement).

In [4]:
from time import perf_counter
import numpy as np

def measure_latency(model, tokenizer, payload, device, generation_args={}):
    input_ids = tokenizer(payload, return_tensors="pt").input_ids.to(device)
    latencies = []
    # Do GPU warm up before benchmarking
    for _ in range(2):
        _ =  model.generate(input_ids, **generation_args)
    # Runs used for measuring the latency
    for _ in range(20):
        start_time = perf_counter()
        _ = model.generate(input_ids, **generation_args)
        latency = perf_counter() - start_time
        latencies.append(latency)

    time_avg_ms = 1000 * np.mean(latencies)
    time_std_ms = 1000 * np.std(latencies)
    time_p95_ms = 1000 * np.percentile(latencies,95)

    return f"P95 latency (ms) - {time_p95_ms}; Average latency (ms) - {time_avg_ms:.2f} +\- {time_std_ms:.2f};", time_p95_ms


<>:21: SyntaxWarning: invalid escape sequence '\-'
<>:21: SyntaxWarning: invalid escape sequence '\-'
/tmp/ipykernel_29554/2141923935.py:21: SyntaxWarning: invalid escape sequence '\-'
  return f"P95 latency (ms) - {time_p95_ms}; Average latency (ms) - {time_avg_ms:.2f} +\- {time_std_ms:.2f};", time_p95_ms


Download the base GPT-Neo 2.7B model in half precision and the related tokenizer from the HF's Hub.

In [17]:
import torch
from transformers import GPTNeoForCausalLM, GPT2Tokenizer, AutoTokenizer, AutoModelForCausalLM

model_id = "EleutherAI/gpt-neo-2.7B"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id,
                                          torch_dtype=torch.float16,
                                          device_map="auto")
print(f"model is loaded on device {model.device.type}")

Loading weights:   0%|          | 0/420 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-2.7B
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
transformer.h.{0...31}.attn.attention.masked_bias | UNEXPECTED |  | 
transformer.h.{0...30}.attn.attention.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model is loaded on device cuda


In [18]:
print(f"model is loaded on device {model.device.type}")

model is loaded on device cuda


In [19]:
model

GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 2560)
    (wpe): Embedding(2048, 2560)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-31): 32 x GPTNeoBlock(
        (ln_1): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=2560, out_features=2560, bias=False)
            (v_proj): Linear(in_features=2560, out_features=2560, bias=False)
            (q_proj): Linear(in_features=2560, out_features=2560, bias=False)
            (out_proj): Linear(in_features=2560, out_features=2560, bias=True)
          )
        )
        (ln_2): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=2560, out_features=10240, bias=True)
          (c_proj)

Do inference with the downloaded model to verify that everything is working as expected.

In [ ]:
example = "The story so far: in the beginning, the universe was created."

input_ids = tokenizer(example, return_tensors="pt").input_ids.to(model.device)
logits = model.generate(input_ids,
                        do_sample=True,
                        num_beams=1,
                        min_length=128,
                        max_new_tokens=128,
                        pad_token_id=50256)

print(f"prediction: \n \n {tokenizer.decode(logits[0].tolist()[len(input_ids[0]):])}")

Perform benchmark for the vanilla model. The previously defined ```measure_latency``` function is used.



In [8]:
generation_args = dict(do_sample=True,
                      max_length=300,
                      pad_token_id=50256,
                      use_cache=True
)
vanilla_results = measure_latency(model, tokenizer, example,
                                  model.device, generation_args)

print(f"Vanilla model: {vanilla_results[0]}")

Vanilla model: P95 latency (ms) - 2686.4592654339503; Average latency (ms) - 2667.70 +\- 12.58;


Let's now optimize the base GPT-Neo 2.7B model for inference on GPU with DeepSpeed. The decision about which of the original model's layers have to be replaced is left to DeepSpeed itself here.

In [20]:
import os

os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '9999'
os.environ['RANK'] = "0"
os.environ['LOCAL_RANK'] = "0"
os.environ['WORLD_SIZE'] = "1"

In [21]:
import deepspeed

ds_model = deepspeed.init_inference(
    model=model,
    mp_size=1,
    dtype=torch.float16,
    replace_method="auto",
    replace_with_kernel_inject=True,
)
print(f"model is loaded on device {ds_model.module.device}")

[2026-06-21 18:56:18,315] [WARNING] [config_utils.py:70:_process_deprecated_field] Config parameter replace_method is deprecated. This parameter is no longer needed, please remove from your call to DeepSpeed-inference
[2026-06-21 18:56:18,315] [WARNING] [config_utils.py:70:_process_deprecated_field] Config parameter mp_size is deprecated use tensor_parallel.tp_size instead
model is loaded on device cuda:0


Print the optimized model's architecture to this cell output to verify that some of the original model's layers have been replaced with DeepSpeed optimized kernel implementations.

In [22]:
ds_model

InferenceEngine(
  (module): GPTNeoForCausalLM(
    (transformer): GPTNeoModel(
      (wte): Embedding(50257, 2560)
      (wpe): Embedding(2048, 2560)
      (drop): Dropout(p=0.0, inplace=False)
      (h): ModuleList(
        (0-31): 32 x DeepSpeedGPTInference(
          (attention): DeepSpeedSelfAttention(
            (qkv_func): QKVGemmOp()
            (score_context_func): SoftmaxContextOp()
            (linear_func): LinearOp()
            (vector_matmul_func): VectorMatMulOp()
          )
          (mlp): DeepSpeedMLP(
            (mlp_gemm_func): MLPGemmOp(
              (pre_rms_norm): PreRMSNormOp()
            )
            (vector_matmul_func): VectorMatMulOp()
            (fused_gemm_gelu): GELUGemmOp()
            (residual_add_func): ResidualAddOp(
              (vector_add): VectorAddOp()
            )
          )
          (layer_norm): LayerNormOp()
        )
      )
      (ln_f): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
    )
    (lm_head): Linear(in_feat

Do inference with the optimized model to verify that everything is working as expected.

In [23]:
input_ids = tokenizer(example, return_tensors="pt").input_ids.to(model.device)
logits = ds_model.generate(input_ids,
                           do_sample=True,
                           num_beams=1,
                           min_length=128,
                           max_new_tokens=128,
                           pad_token_id=50256,
                           use_cache=False
                          )
print(tokenizer.decode(logits[0].tolist()))

The story so far: in the beginning, the universe was created. Everything that exists is just one more part of it. No part exists for itself. The whole universe exists to bring everything else into being.

What a wonderful, simple story. We can think of this also as:

Everything is connected,

There is no me without you.

But the reason we can relate this idea to so many different areas of our nature is that this is very true for our relationships as well. In fact we have this amazing quality when it comes to relationships.

Even if we know each other, we still don’t know each other. Just in case I feel lonely


Perform now benchmark for the DeepSpeed optimized model. The previously defined ```measure_latency``` function is used.

In [ ]:
generation_args = dict(do_sample=True,
                       max_length=300,
                       pad_token_id=50256,
                       use_cache=False)
ds_results = measure_latency(ds_model, tokenizer, example,
                             ds_model.module.device, generation_args)

print(f"DeepSpeed model: {ds_results[0]}")
#Vanilla model: P95 latency (ms) - 2686.4592654339503; Average latency (ms) - 2667.70 +\- 12.58;

DeepSpeed model: P95 latency (ms) - 2200.3251655143686; Average latency (ms) - 2187.44 +\- 8.54;
